# EuroCoin Pipeline Demo

This notebook demonstrates the trained multi-stage pipeline with the published weights stored in
`project_root/model_weights`.

It keeps the presentation intentionally simple:

- input image on the left
- final annotated output on the right
- denomination labels drawn directly on the predicted boxes


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import os

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import resnet18
from ultralytics import YOLO
import yaml


@dataclass(frozen=True)
class ProjectPaths:
    project_root: Path

    @property
    def model_weights_dir(self) -> Path:
        return self.project_root / "model_weights"

    @property
    def ml_pipeline_dir(self) -> Path:
        return self.project_root / "ml_pipeline"

    def stage_dataset_dir(self, stage_name: str) -> Path:
        return self.ml_pipeline_dir / "datasets" / stage_name

    def stage_weight_dir(self, stage_name: str) -> Path:
        return self.model_weights_dir / stage_name


class ProjectLocator:
    def find(self) -> ProjectPaths:
        candidates: list[Path] = []

        env_root = os.getenv("EUROCOIN_VISION_ROOT")
        if env_root:
            candidates.append(Path(env_root).expanduser().resolve())

        cwd = Path.cwd().resolve()
        candidates.extend([cwd, *cwd.parents])

        for candidate in candidates:
            if self._looks_like_project_root(candidate):
                return ProjectPaths(candidate)

        raise FileNotFoundError(
            "Could not locate the project root. "
            "Run the notebook from inside the eurocoin-vision repo or set EUROCOIN_VISION_ROOT."
        )

    def _looks_like_project_root(self, candidate: Path) -> bool:
        return (candidate / "ml_pipeline").exists() and (candidate / "model_weights").exists()


def find_latest_file(root_dir: Path, pattern: str) -> Path:
    if not root_dir.exists():
        raise FileNotFoundError(f"Missing directory: {root_dir}")

    matches = sorted(
        (path for path in root_dir.rglob(pattern) if path.is_file()),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )
    if not matches:
        raise FileNotFoundError(f"No files matching {pattern!r} under {root_dir}")
    return matches[0]


paths = ProjectLocator().find()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Project root: {paths.project_root}")
print(f"Device: {device}")


In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


@dataclass
class ClassifierCheckpoint:
    model: nn.Module
    class_names: list[str]
    image_size: int
    device: torch.device

    def __post_init__(self) -> None:
        self._transform = transforms.Compose(
            [
                transforms.Resize((self.image_size, self.image_size)),
                transforms.ToTensor(),
                transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ]
        )

    def predict(self, image: Image.Image) -> tuple[str, float]:
        tensor = self._transform(image).unsqueeze(0).to(self.device)
        with torch.no_grad():
            logits = self.model(tensor)
            probabilities = F.softmax(logits, dim=1)[0]

        predicted_index = int(probabilities.argmax().item())
        return self.class_names[predicted_index], float(probabilities[predicted_index].item())


def load_classifier_checkpoint(checkpoint_path: Path, device: torch.device) -> ClassifierCheckpoint:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    class_names = list(checkpoint["class_names"])
    image_size = int(checkpoint.get("image_size", 224))

    model = resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, len(class_names))
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    return ClassifierCheckpoint(
        model=model,
        class_names=class_names,
        image_size=image_size,
        device=device,
    )


stage1_weights_path = find_latest_file(paths.stage_weight_dir("stage1"), "*.pt")
stage2_weights_path = find_latest_file(paths.stage_weight_dir("stage2"), "*.pt")
stage3_manifest_path = find_latest_file(paths.stage_weight_dir("stage3"), "*.yaml")

stage3_manifest = yaml.safe_load(stage3_manifest_path.read_text(encoding="utf-8"))
stage3_material_names = list(stage3_manifest["mapping"].keys())
stage3_weights_by_material = {
    material_name: find_latest_file(paths.stage_weight_dir(f"stage3_{material_name}"), "*.pt")
    for material_name in stage3_material_names
}

stage1_detector = YOLO(str(stage1_weights_path))
stage2_classifier = load_classifier_checkpoint(stage2_weights_path, device)
stage3_classifiers = {
    material_name: load_classifier_checkpoint(weights_path, device)
    for material_name, weights_path in stage3_weights_by_material.items()
}

print(f"Stage 1 weights: {stage1_weights_path}")
print(f"Stage 2 weights: {stage2_weights_path}")
print(f"Stage 3 manifest: {stage3_manifest_path}")
for material_name, weights_path in stage3_weights_by_material.items():
    print(f"Stage 3 ({material_name}) weights: {weights_path}")


In [ ]:
@dataclass(frozen=True)
class CoinDetection:
    box_xyxy: tuple[int, int, int, int]
    detection_confidence: float
    material: str
    denomination: str


@dataclass(frozen=True)
class PipelinePrediction:
    image_path: Path
    input_image: Image.Image
    detections: list[CoinDetection]


class EuroCoinPipeline:
    def __init__(
        self,
        detector: YOLO,
        material_classifier: ClassifierCheckpoint,
        denomination_classifiers: dict[str, ClassifierCheckpoint],
    ) -> None:
        self._detector = detector
        self._material_classifier = material_classifier
        self._denomination_classifiers = denomination_classifiers

    def predict(
        self,
        image_path: Path,
        confidence_threshold: float = 0.25,
        iou_threshold: float = 0.45,
        padding_ratio: float = 0.05,
    ) -> PipelinePrediction:
        image_path = Path(image_path)
        image = Image.open(image_path).convert("RGB")
        image_array = np.array(image)

        result = self._detector.predict(
            source=image_array,
            conf=confidence_threshold,
            iou=iou_threshold,
            verbose=False,
        )[0]

        detections: list[CoinDetection] = []
        if result.boxes is None or len(result.boxes) == 0:
            return PipelinePrediction(image_path=image_path, input_image=image, detections=detections)

        boxes = result.boxes.xyxy.detach().cpu().numpy()
        confidences = result.boxes.conf.detach().cpu().numpy()
        width, height = image.size

        for box_xyxy, confidence in zip(boxes, confidences):
            left, top, right, bottom = self._expand_box(
                box_xyxy=box_xyxy,
                image_width=width,
                image_height=height,
                padding_ratio=padding_ratio,
            )

            crop = image.crop((left, top, right, bottom))
            material_name, _ = self._material_classifier.predict(crop)
            denomination_name, _ = self._denomination_classifiers[material_name].predict(crop)

            detections.append(
                CoinDetection(
                    box_xyxy=(left, top, right, bottom),
                    detection_confidence=float(confidence),
                    material=material_name,
                    denomination=denomination_name,
                )
            )

        detections.sort(key=lambda detection: (detection.box_xyxy[1], detection.box_xyxy[0]))
        return PipelinePrediction(image_path=image_path, input_image=image, detections=detections)

    def _expand_box(
        self,
        box_xyxy: np.ndarray,
        image_width: int,
        image_height: int,
        padding_ratio: float,
    ) -> tuple[int, int, int, int]:
        left, top, right, bottom = [float(value) for value in box_xyxy]
        box_width = right - left
        box_height = bottom - top
        pad_x = box_width * padding_ratio
        pad_y = box_height * padding_ratio

        expanded_left = max(0, int(round(left - pad_x)))
        expanded_top = max(0, int(round(top - pad_y)))
        expanded_right = min(image_width, int(round(right + pad_x)))
        expanded_bottom = min(image_height, int(round(bottom + pad_y)))

        return expanded_left, expanded_top, expanded_right, expanded_bottom


def format_label(raw_label: str) -> str:
    return raw_label.replace("_", " ")


def show_prediction(prediction: PipelinePrediction) -> None:
    figure, axes = plt.subplots(1, 2, figsize=(16, 8))

    axes[0].imshow(prediction.input_image)
    axes[0].set_title(f"Input: {prediction.image_path.name}")
    axes[0].axis("off")

    axes[1].imshow(prediction.input_image)
    axes[1].set_title("Output")
    axes[1].axis("off")

    for detection in prediction.detections:
        left, top, right, bottom = detection.box_xyxy
        rectangle = Rectangle(
            (left, top),
            right - left,
            bottom - top,
            fill=False,
            linewidth=2.5,
            edgecolor="#00A86B",
        )
        axes[1].add_patch(rectangle)
        axes[1].text(
            left,
            max(6, top - 8),
            format_label(detection.denomination),
            color="white",
            fontsize=10,
            bbox={
                "facecolor": "#008E5A",
                "edgecolor": "none",
                "pad": 2,
                "alpha": 0.9,
            },
        )

    if not prediction.detections:
        axes[1].text(
            0.5,
            0.5,
            "No coins detected",
            transform=axes[1].transAxes,
            ha="center",
            va="center",
            fontsize=14,
            color="crimson",
        )

    plt.tight_layout()
    plt.show()


pipeline = EuroCoinPipeline(
    detector=stage1_detector,
    material_classifier=stage2_classifier,
    denomination_classifiers=stage3_classifiers,
)


In [ ]:
sample_image_dir = paths.stage_dataset_dir("stage3") / "images" / "test"
sample_image_paths = sorted(
    path for path in sample_image_dir.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

print(f"Sample image directory: {sample_image_dir}")
print(f"Available sample images: {len(sample_image_paths)}")
sample_image_paths[:5]


In [ ]:
images_to_show = sample_image_paths[:3]

for image_path in images_to_show:
    prediction = pipeline.predict(image_path)
    show_prediction(prediction)


In [ ]:
# Optional: run the same visualization on your own image.
# Set custom_image_path to a relative path from the project root or to an absolute path.

custom_image_path = None

if custom_image_path:
    candidate_path = Path(custom_image_path)
    if not candidate_path.is_absolute():
        candidate_path = (paths.project_root / candidate_path).resolve()

    prediction = pipeline.predict(candidate_path)
    show_prediction(prediction)
